<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z323_RegresionPanel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresion Panel — Prediccion directa a t+2

## La idea

HAR y MARS ajustan un modelo **por producto** con ~24 observaciones cada uno. Esta alternativa hace lo opuesto: convierte las 780 series en una **tabla panel** y entrena **un solo modelo global** con ~18.000 filas.

```
HAR / MARS:
  780 modelos × 24 obs = 24 obs por modelo

Regresion Panel:
  1 modelo × 18.000 obs = aprende patrones entre productos
```

## Ventaja clave: prediccion directa a t+2

HAR predice t+1 primero, luego usa esa prediccion para predecir t+2 (recursive forecasting). Cada paso acumula error.

Acá el target es **directamente `tn_{t+2}`** — sin pasos intermedios, sin error acumulado.

## Conexion con mediciones repetidas

Es un modelo de **datos de panel** (panel data) — la misma estructura que mediciones repetidas en estadistica:
- Cada producto es una unidad observada múltiples veces
- El modelo captura tanto la variacion **entre productos** como la variacion **dentro de cada producto en el tiempo**
- El efecto fijo por producto (`product_id` como feature) actua como intercepto individual

## Modelos que probamos

1. **Regresion lineal** (OLS) — baseline interpretable
2. **Ridge** — regularizacion para estabilizar coeficientes colineales
3. **Lasso** — seleccion automatica de features relevantes
4. **LightGBM** — captura no linealidades y efectos de interaccion

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle lightgbm

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
  import os
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  os.system(comando)

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento': 'RegresionPanel-01',
  'kaggle_competition': 'labo-iii-2026-rosario',
  'semilla_primigenia': 102191,
  # modelo a usar: 'OLS', 'Ridge', 'Lasso', 'LightGBM'
  'modelo': 'Ridge'
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Preparacion de datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")

tb_ventas = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])
print(f"{tb_ventas.height} filas, {tb_ventas['product_id'].n_unique()} productos")

# 3  Construccion de la tabla panel

Para cada producto × tiempo `t` construimos una fila con:

| Feature | Descripcion |
|---|---|
| `lag1` | tn_{t-1} |
| `lag2` | tn_{t-2} |
| `lag3` | tn_{t-3} |
| `mean_3m` | promedio tn_{t-3:t-1} |
| `mean_6m` | promedio tn_{t-6:t-1} |
| `mean_12m` | promedio tn_{t-12:t-1} |
| `mes` | mes del año de t+2 (1-12) — captura estacionalidad |
| `product_id` | efecto fijo por producto |
| **target** | `tn_{t+2}` — prediccion directa, sin pasos intermedios |

In [ ]:
def build_panel(tb: pl.DataFrame, periodos_todos: list) -> pl.DataFrame:
    """
    Construye la tabla panel: una fila por (producto, tiempo t)
    con features de lags y target = tn_{t+2}.
    Solo genera filas donde t+2 existe en los datos.
    """
    filas = []

    for pid in tb["product_id"].unique().to_list():
        serie_df = tb.filter(pl.col("product_id") == pid).sort("periodo")
        periodos = serie_df["periodo"].to_list()
        tn = serie_df["tn"].to_numpy().astype(float)
        T = len(tn)

        # necesitamos al menos 12 de historia + 2 de horizonte
        for t in range(12, T - 2):
            # periodo del target (t+2)
            periodo_target = periodos[t + 2]
            mes_target = int(str(periodo_target)[4:6])

            filas.append({
                'product_id': pid,
                'periodo_t':  periodos[t],
                'lag1':       tn[t - 1],
                'lag2':       tn[t - 2],
                'lag3':       tn[t - 3],
                'mean_3m':    tn[t-3:t].mean(),
                'mean_6m':    tn[t-6:t].mean(),
                'mean_12m':   tn[t-12:t].mean(),
                'mes':        mes_target,
                'tn_t2':      tn[t + 2]   # target
            })

    return pl.DataFrame(filas)


periodos_todos = tb_ventas["periodo"].unique().sort().to_list()
tb_panel = build_panel(tb_ventas, periodos_todos)

print(f"Tabla panel: {tb_panel.height} filas x {tb_panel.width} columnas")
display(tb_panel.head(5))

# 4  Split train / validacion

El split es **temporal**: entrenamos con los periodos mas viejos y validamos con los mas recientes.

Nunca mezclamos futuro en el entrenamiento — es el mismo principio que `num_val_windows` en AutoGluon.

In [ ]:
# split temporal: ultimos 6 periodos de entrenamiento como validacion
periodo_corte = tb_panel["periodo_t"].sort(descending=True).unique()[6]

tb_train = tb_panel.filter(pl.col("periodo_t") <= periodo_corte)
tb_val   = tb_panel.filter(pl.col("periodo_t") >  periodo_corte)

print(f"Train: {tb_train.height} filas")
print(f"Val  : {tb_val.height} filas")

FEATURES = ['lag1', 'lag2', 'lag3', 'mean_3m', 'mean_6m', 'mean_12m', 'mes', 'product_id']
TARGET   = 'tn_t2'

X_train = tb_train.select(FEATURES).to_numpy()
y_train = tb_train[TARGET].to_numpy()
X_val   = tb_val.select(FEATURES).to_numpy()
y_val   = tb_val[TARGET].to_numpy()

# 5  Entrenamiento

Cambiando `PARAM['modelo']` en la celda de configuracion podés probar OLS, Ridge, Lasso o LightGBM sin tocar nada mas.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

if PARAM['modelo'] == 'OLS':
    modelo = LinearRegression()
    modelo.fit(X_train_sc, y_train)

elif PARAM['modelo'] == 'Ridge':
    # RidgeCV busca el mejor alpha automaticamente por cross-validation
    modelo = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
    modelo.fit(X_train_sc, y_train)
    print(f"Ridge alpha optimo: {modelo.alpha_}")

elif PARAM['modelo'] == 'Lasso':
    modelo = LassoCV(cv=5, random_state=PARAM['semilla_primigenia'], max_iter=5000)
    modelo.fit(X_train_sc, y_train)
    print(f"Lasso alpha optimo: {modelo.alpha_}")

elif PARAM['modelo'] == 'LightGBM':
    modelo = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=PARAM['semilla_primigenia'],
        verbose=-1
    )
    # LightGBM no necesita scaling
    modelo.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

# score en validacion
if PARAM['modelo'] == 'LightGBM':
    pred_val = modelo.predict(X_val)
else:
    pred_val = modelo.predict(X_val_sc)

rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
print(f"RMSE validacion ({PARAM['modelo']}): {rmse_val:.4f}")

## 5.1 Importancia de features (Ridge / Lasso / LightGBM)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

if PARAM['modelo'] in ('OLS', 'Ridge', 'Lasso'):
    coefs = modelo.coef_
    ax.barh(FEATURES, coefs, color='steelblue')
    ax.axvline(0, color='red', linewidth=0.8)
    ax.set_title(f'Coeficientes {PARAM["modelo"]} (estandarizados)')

elif PARAM['modelo'] == 'LightGBM':
    imp = modelo.feature_importances_
    ax.barh(FEATURES, imp, color='steelblue')
    ax.set_title('Importancia de features — LightGBM')

plt.tight_layout()
plt.show()

# 6  Prediccion para 202002

Para predecir `202002` usamos como features los valores conocidos hasta `201912`:
- `lag1` = tn de `201911`
- `lag2` = tn de `201910`
- `lag3` = tn de `201909`
- `mean_3m`, `mean_6m`, `mean_12m` calculados sobre los ultimos meses conocidos
- `mes` = 2 (febrero)

No hace falta predecir `202001` primero — el modelo fue entrenado para saltar directamente a t+2.

In [ ]:
productos = tb_apredecir["product_id"].to_list()
resultados = []

def safe_mean(arr):
    return float(arr.mean()) if len(arr) > 0 else 0.0

for pid in productos:
    serie = (
        tb_ventas.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    t = len(serie) - 1

    fila = np.array([[
        serie[t - 1] if t >= 1 else 0.0,
        serie[t - 2] if t >= 2 else 0.0,
        serie[t - 3] if t >= 3 else 0.0,
        safe_mean(serie[max(0, t-3):t]),
        safe_mean(serie[max(0, t-6):t]),
        safe_mean(serie[max(0, t-12):t]),
        2,
        pid
    ]])

    if PARAM['modelo'] == 'LightGBM':
        pred = float(modelo.predict(fila)[0])
    else:
        pred = float(modelo.predict(scaler.transform(fila))[0])

    pred = max(pred, 0.0)
    resultados.append({'product_id': pid, 'tn': pred})

tb_final = pl.DataFrame(resultados)
display(tb_final.head(10))
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

# 7  Submit a Kaggle

In [ ]:
archivo = f"RegresionPanel_{PARAM['modelo']}.csv"
mensaje = f"Panel directo t+2 {PARAM['modelo']} features HAR + mes + product_id"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)

# 8  Que probar

| Cambio | Donde | Por que |
|---|---|---|
| `'modelo': 'LightGBM'` | PARAM | Captura no linealidades y product_id como categorica |
| `'modelo': 'Lasso'` | PARAM | Seleccion automatica — puede descartar features inutiles |
| Agregar `lag6`, `lag12` individuales | `build_panel` | Mas señal del pasado lejano |
| Agregar `cv = coef_variacion` | `build_panel` | Volatilidad del producto como feature |
| Transformar con `log1p` en train y deshacer en pred | antes del fit | Estabiliza series con outliers |
| `product_id` como one-hot en vez de numerico | antes del fit | Mejor representacion del efecto fijo |